# 00 -- Environment & data check

Goal of this notebook: in **5 minutes**, confirm that

1. `metapulsar` and the heavy optional dependencies (`enterprise`, `pint`, `libstempo`) are importable;
2. the IPTA-DR2 data needed by notebooks 02 and 03 is reachable;
3. a single MetaPulsar can be built end-to-end with default settings on a small subset of pulsars.

If anything fails here, fix it before doing the tutorial

## Notebook flow

* `00_setup.ipynb` (this notebook) -- environment + data test, persists `DATA_ROOT_STR` and `PULSAR_SUBSET`.
* `01_frankenstat_composite.ipynb` -- David Wright's *FrankenStat* composite strategy on **simulated** data (independent of this notebook; uses [`./frankenstat/`](frankenstat/)).
* `02_metapulsar_consistent.ipynb` -- MetaPulsar's file/layout-discovery tooling and the *consistent* combination strategy on the **real IPTA-DR2** release.

## Tutorial pulsars

We pin two pulsars:

* **J1853+1303** -- EPTA dr2 (Tempo2, 101 TOAs) + NANOGrav 9y (PINT, 1369 TOAs).
* **B1953+29**   -- EPTA dr2 (Tempo2, 179 TOAs) + NANOGrav 9y (PINT, 1302 TOAs).

These were chosen to exercise the **PINT + Tempo2** code path in MetaPulsar while keeping the total TOA count small (~3k vs, say, ~26k for J0613-0200 / J1713+0747). This makes things somewhat fast-ish. Swap them for your own pulsars by editing `PULSAR_SUBSET` below; everything downstream scales to N pulsars where needed.

## Step 1 -- Environment check

Hard requirement: `metapulsar` itself, plus `enterprise` and `pint`. `libstempo`, `discovery`, `tensiometer`, and `getdist` are optional and only used in specific notebooks; we just print whether they are present.

In [7]:
import sys
import warnings
from importlib import import_module
from importlib.metadata import (
    PackageNotFoundError,
    packages_distributions,
    version as pkg_version,
)
from pathlib import Path

import loguru

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# This is a temporary hack
# will be fixed by PR 10 in MetaPulsar
def quiet_loguru(level: str = "WARNING") -> None:
    """Tame loguru so notebook output is readable."""
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)


quiet_loguru()

# Some packages have an import name that differs from their PyPI
# distribution name (e.g. `enterprise` -> `enterprise-pulsar`).
# `packages_distributions()` gives us the right mapping at runtime.
_import_to_dist = packages_distributions()


def _version_for_import(import_name: str) -> str:
    for dist_name in _import_to_dist.get(import_name, [import_name]):
        try:
            return pkg_version(dist_name)
        except PackageNotFoundError:
            continue
    return "(no distribution metadata)"


# Show versions of all packages
print(f"Python: {sys.version.split()[0]}")
for mod in ("metapulsar", "enterprise", "pint", "libstempo", "discovery", "getdist", "tensiometer"):
    try:
        import_module(mod)
    except Exception as exc:
        print(f"  {mod:<14s} MISSING ({exc.__class__.__name__})")
        continue
    print(f"  {mod:<14s} {_version_for_import(mod)}")

Python: 3.12.3
  metapulsar     0.9.6.post1.dev1
  enterprise     3.4.5.dev71+g64780f449
  pint           1.1.4
  libstempo      2.5.1
  discovery      0.5
  getdist        1.7.6
  tensiometer    1.1.0


## Step 2 -- Resolve `DATA_ROOT`

The IPTA-DR2 release lives under `../../data/ipta-dr2/` as a **git submodule** (<https://gitlab.com/IPTA/DR2.git>) of MetaPulsar. On a fresh `git clone` of `metapulsar`, the directory exists but is empty until you initialise the submodule. From the repo root run

```bash
git submodule update --init data/ipta-dr2
```

The cell below verifies that `data/ipta-dr2/` is populated and gives a copy-pasteable fix-up command if it is not.

In [8]:
DATA_ROOT = Path("../../data/ipta-dr2").resolve()

if not (DATA_ROOT / "EPTA_v2.2").exists():
    raise FileNotFoundError(
        f"{DATA_ROOT} is missing or empty -- the IPTA-DR2 git submodule is not "
        f"initialised. From the repo root run:\n"
        "    git submodule update --init data/ipta-dr2\n"
    )

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"  EPTA_v2.2/   present: {(DATA_ROOT / 'EPTA_v2.2').exists()}")
print(f"  PPTA_dr1dr2/ present: {(DATA_ROOT / 'PPTA_dr1dr2').exists()}")
print(f"  NANOGrav_9y/ present: {(DATA_ROOT / 'NANOGrav_9y').exists()}")

DATA_ROOT = /workspaces/metapulsar/data/ipta-dr2
  EPTA_v2.2/   present: True
  PPTA_dr1dr2/ present: True
  NANOGrav_9y/ present: True


## Step 3 -- Minimal discovery for the test

MetaPulsar's `discover_layout` + `discover_files` pipeline finds `.par` and `.tim` files inside each PTA release. We only run enough of it here to feed a single-pulsar `create_metapulsar` call -- the full layout-discovery walkthrough (regex tweaks, `pta_summary`, coordinate matching, ...) lives in `02_metapulsar_consistent.ipynb`.

In [9]:
from metapulsar import combine_layouts, discover_files, discover_layout

quiet_loguru()

epta_layout = discover_layout(str(DATA_ROOT / "EPTA_v2.2"), name="EPTA dr2", verbose=False)
ppta_layout = discover_layout(str(DATA_ROOT / "PPTA_dr1dr2"), name="PPTA dr1dr2", verbose=False)
nanograv_layout = discover_layout(str(DATA_ROOT / "NANOGrav_9y"), name="NANOGrav 9y", verbose=False)

combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)
file_data = discover_files(combined_layout, verbose=False)

print("Discovered PTAs:", list(file_data.keys()))
for pta, files in file_data.items():
    print(f"  {pta:<15s} -> {len(files):>3d} pulsars")

Discovered PTAs: ['EPTA dr2', 'NANOGrav 9y', 'PPTA dr1dr2']
  EPTA dr2        ->  42 pulsars
  NANOGrav 9y     ->  37 pulsars
  PPTA dr1dr2     ->  18 pulsars


## Step 4 -- Pin `PULSAR_SUBSET` and test of `create_metapulsar`

Build a single MetaPulsar (J1853+1303) with the default `consistent` strategy. If this cells complete in about a minute on the smaller pulsar, the heavier two-pulsar runs in notebooks 02 / 03 will also be tractable on a laptop just fine

In [10]:
from metapulsar import create_metapulsar, filter_file_data_by_pulsars

quiet_loguru()

PULSAR_SUBSET = ["J1853+1303", "B1953+29"]
filtered_data = filter_file_data_by_pulsars(file_data, PULSAR_SUBSET)

for pta, files in filtered_data.items():
    matched = sorted({Path(f["par"]).name for f in files})
    print(f"{pta:<15s} -> {matched}")

2026-04-24 07:43:27.274 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:588 - [J0931-1902] Missing PMELONG/PMELAT or POSEPOCH/PEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.
2026-04-24 07:43:27.310 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:588 - [J1832-0836] Missing PMELONG/PMELAT or POSEPOCH/PEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.
2026-04-24 07:43:27.341 | WARNING  | metapulsar.position_helpers:_extract_equatorial_coordinates_optimized:542 - [J1022+1001] Missing PMRA/PMDEC or POSEPOCH/PEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.
2026-04-24 07:43:27.346 | WARNING  | metapulsar.position_helpers:_extract_equatorial_coordinates_optimized:542 - [J1730-2304] Missing PMRA/PM

In [11]:
quiet_loguru()

# `filter_file_data_by_pulsars` accepts either a single pulsar name or a list,
# and resolves J/B name aliases via coordinate-based matching.
test_target = "J1853+1303"
single_pulsar_data = filter_file_data_by_pulsars(file_data, test_target)

test_mp = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
)

print(f"Built MetaPulsar  : {test_mp.name}")
print(f"  TOAs            : {len(test_mp.toas)}")
print(f"  PTAs combined   : {list(test_mp._pulsars.keys())}")
print(f"  Strategy        : {test_mp.combination_strategy}")

2026-04-24 07:43:27.895 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.
2026-04-24 07:43:27.902 | WARNING  | pint.models.model_builder:__call__:224 - UNITS is not specified. Assuming TDB...
2026-04-24 07:43:27.997 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.


/opt/venvs/pta/lib/python3.12/site-packages/enterprise/signals/utils.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 


2026-04-24 07:43:40.626 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.




Results for PSR J1853+1303


RMS pre-fit residual = 0.000 (us), RMS post-fit residual = 14.531 (us)
Fit Chisq = 0	Chisqr/nfree = 0.00/0 = nan	pre/post = 0
Number of fit parameters: 0
Number of points in fit = 0
Offset: 0 1 offset_e*sqrt(n) = 0 n = 0


PARAMETER       Pre-fit                   Post-fit                  Uncertainty   Difference   Fit
---------------------------------------------------------------------------------------------------
RAJ (rad)       4.94781344420877          4.94781344420877          0             0             Y
RAJ (hms)       18:53:57.3187611           18:53:57.3187611         0             0            
DECJ (rad)      0.227979121332348         0.227979121332348         0             0             Y
DECJ (dms)      +13:03:44.06929           +13:03:44.06929           0             0            


2026-04-24 07:43:52.357 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.


F0 (s^-1)       244.391377820396          244.391377820396          0             0             Y
F1 (s^-2)       -5.20456761235233e-16     -5.20456761235233e-16     0             0             Y
PEPOCH (MJD)    54999.9998161704          54999.9998161704          0             0             N
POSEPOCH (MJD)  54999.9998161704          54999.9998161704          0             0             N
DMEPOCH (MJD)   55000                     55000                     0             0             N
DM (cm^-3 pc)   30.5609849277718          30.5609849277718          0             0             Y
DM1 (cm^-3 pc y 0                         0                         0             0             Y
DM2 (cm^-3 pc y 0                         0                         0             0             Y
PMRA (mas/yr)   -1.73450001334112         -1.73450001334112         0             0             Y
PMDEC (mas/yr)  -2.81841594816981         -2.81841594816981         0             0             Y
PB (d)          115.

2026-04-24 07:43:52.635 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.
Built MetaPulsar  : J1853+1303
  TOAs            : 1470
  PTAs combined   : ['EPTA dr2', 'NANOGrav 9y']
  Strategy        : consistent


## Step 5 -- Persist session variables

`%store` writes these into the IPython profile so notebooks 02 and 03 can `%store -r` them without rerunning discovery. Notebook 01 (FrankenStat on simulated data) is independent of these and does not need them.

In [12]:
DATA_ROOT_STR = str(DATA_ROOT)
%store DATA_ROOT_STR
%store PULSAR_SUBSET
print("\nSetup complete. Continue with 01_frankenstat_composite.ipynb (independent),")
print("or skip directly to 02_metapulsar_consistent.ipynb for the MetaPulsar walkthrough.")

Stored 'DATA_ROOT_STR' (str)
Stored 'PULSAR_SUBSET' (list)

Setup complete. Continue with 01_frankenstat_composite.ipynb (independent),
or skip directly to 02_metapulsar_consistent.ipynb for the MetaPulsar walkthrough.
